In [ ]:
## Phase 2

## Phase 2

In [ ]:
!pip uninstall -y trl transformers peft accelerate bitsandbytes


Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
Found existing installation: peft 0.18.1
Uninstalling peft-0.18.1:
  Successfully uninstalled peft-0.18.1
Found existing installation: accelerate 1.13.0
Uninstalling accelerate-1.13.0:
  Successfully uninstalled accelerate-1.13.0


In [ ]:
# Install compatible stack
!pip install -U --quiet \
transformers==4.45.2 \
peft==0.11.1 \
trl==0.9.6 \
accelerate \
bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 119.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.8/245.8 kB 29.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 41.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 43.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 49.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 124.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 120.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 181.9/181.9 kB 21.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviou

In [ ]:
import transformers,trl,peft
import subprocess
import sys

# Verify the installed transformers version
try:
    result = subprocess.run([sys.executable, '-m', 'pip', 'show', 'transformers'], capture_output=True, text=True)
    if result.returncode == 0:
        print("--- Installed transformers package info ---")
        print(result.stdout)
    else:
        print("Could not get pip show transformers info.")
except Exception as e:
    print(f"Error checking pip show transformers: {e}")

print(f"--- Loaded module versions ---")
print(f"transformers.__version__: {transformers.__version__}")
print(f"trl.__version__: {trl.__version__}")
print(f"peft.__version__: {peft.__version__}")

--- Installed transformers package info ---
Name: transformers
Version: 4.45.2
Summary: State-of-the-art Machine Learning for JAX, PyTorch and TensorFlow
Home-page: https://github.com/huggingface/transformers
Author: The Hugging Face team (past and future) with the help of all our contributors (https://github.com/huggingface/transformers/graphs/contributors)
Author-email: transformers@huggingface.co
License: Apache 2.0 License
Location: /usr/local/lib/python3.12/dist-packages
Requires: filelock, huggingface-hub, numpy, packaging, pyyaml, regex, requests, safetensors, tokenizers, tqdm
Required-by: peft, sentence-transformers, trl

--- Loaded module versions ---
transformers.__version__: 4.45.2
trl.__version__: 0.9.6
peft.__version__: 0.11.1


# Google Colab Environment

In [ ]:
from google.colab import userdata, drive
import os

# Retrieve the HF_TOKEN from Colab secrets
hf_token = userdata.get('HF_TOKEN')
os.environ['HF_TOKEN'] = hf_token
print("HF_TOKEN loaded from Colab secrets and set as environment variable.")

# Mount Drive
drive.mount('/content/drive')
PROJECT_DIR = "/content/drive/MyDrive/Datasci_266_NLP/Final_Project/CinePile_Phase2"
os.makedirs(PROJECT_DIR, exist_ok=True)



HF_TOKEN loaded from Colab secrets and set as environment variable.


In [ ]:
# # ============================================================
# # Phase 2: PEFT Fine-tuning on CinePile (Text-Only)
# # Methods: QLoRA, Prefix Tuning, ReFT (LoReFT)
# # !pip install transformers datasets accelerate bitsandbytes peft trl pyreft
# # ============================================================

# import gc
# import os
# import torch
# import pandas as pd
# import numpy as np
# from abc import ABC, abstractmethod
# from dataclasses import dataclass, field
# from typing import Optional
# from datasets import load_dataset
# from transformers import (
#     AutoTokenizer,
#     AutoModelForCausalLM,
#     TrainingArguments,
#     BitsAndBytesConfig,
#     DataCollatorWithPadding,
# )
# from peft import (
#     LoraConfig,
#     PrefixTuningConfig,
#     get_peft_model,
#     TaskType,
#     PeftModel,
# )
# from trl import SFTTrainer, SFTConfig
# from tqdm import tqdm


# # ============================================================
# # 1. 配置类
# # ============================================================
# @dataclass
# class Phase2Config:
#     # --- 数据集 ---
#     dataset_name: str          = 'tomg-group-umd/cinepile'
#     train_split: str           = 'train'
#     test_split: str            = 'test'
#     max_train_samples: Optional[int] = 2000   # 调试用；None = 完整训练集
#     max_test_samples: Optional[int]  = 500    # 调试用；None = 完整测试集

#     # --- 基础模型（只选一个，节省显存）---
#     base_model_id: str         = 'meta-llama/Llama-3.1-8B-Instruct'

#     # --- 输入格式 ---
#     max_scene_length: int      = 1000   # 训练时截短以节省显存
#     max_seq_length: int        = 1024   # SFTTrainer 最大序列长度

#     # --- 训练通用参数 ---
#     num_train_epochs: int      = 2
#     per_device_train_batch_size: int = 2
#     gradient_accumulation_steps: int = 4   # 等效 batch size = 8
#     learning_rate: float       = 2e-4
#     warmup_ratio: float        = 0.05
#     lr_scheduler_type: str     = 'cosine'
#     fp16: bool                 = False
#     bf16: bool                 = True
#     logging_steps: int         = 20
#     save_steps: int            = 200
#     output_base_dir: str       = './phase2_outputs'

#     # --- QLoRA 参数 ---
#     lora_r: int                = 16
#     lora_alpha: int            = 32
#     lora_dropout: float        = 0.05
#     lora_target_modules: list  = field(default_factory=lambda: [
#         "q_proj", "k_proj", "v_proj", "o_proj",
#         "gate_proj", "up_proj", "down_proj"
#     ])

#     # --- Prefix Tuning 参数 ---
#     prefix_num_virtual_tokens: int = 20   # 也可以试 50

#     # --- ReFT (LoReFT) 参数 ---
#     reft_rank: int             = 4
#     reft_layers: list          = field(default_factory=lambda: [8, 16, 24])

#     # --- 评估 ---
#     max_new_tokens: int        = 5
#     output_csv: str            = 'phase2_results.csv'

#     # --- 类别映射（与 Phase 1 一致）---
#     category_map: dict = field(default_factory=lambda: {
#         'TEMP': 'Temporal',
#         'CRD':  'Character and',
#         'NPA':  'Narrative and',
#         'STA':  'Setting and',
#         'TH':   'Theme Exploration',
#     })


# # ============================================================
# # 2. 数据集（复用 Phase 1 的 answer_key 修复逻辑）
# # ============================================================
# class CinePileDataset:
#     LETTERS = ['A', 'B', 'C', 'D', 'E']

#     def __init__(self, config: Phase2Config):
#         self.config = config
#         self.train_data = self._load(config.train_split, config.max_train_samples)
#         self.test_data  = self._load(config.test_split,  config.max_test_samples)
#         print(f"[Dataset] Train: {len(self.train_data)} | Test: {len(self.test_data)}")

#     @classmethod
#     def _answer_text_to_letter(cls, answer_text: str, choices: list) -> str:
#         for i, c in enumerate(choices):
#             if answer_text.strip() == c.strip():
#                 return cls.LETTERS[i]
#         for i, c in enumerate(choices):
#             if answer_text.strip() in c or c.strip() in answer_text:
#                 return cls.LETTERS[i]
#         return 'A'

#     @staticmethod
#     def _normalize_hard_split(raw) -> bool:
#         if isinstance(raw, bool):
#             return raw
#         return str(raw).strip().lower() == 'true'

#     def _load(self, split: str, max_samples: Optional[int]) -> list:
#         raw = load_dataset(self.config.dataset_name, split=split)
#         samples = []
#         for ex in raw:
#             letter = self._answer_text_to_letter(ex['answer_key'], ex['choices'])
#             samples.append({
#                 'movie_scene':       ex['movie_scene'],
#                 'question':          ex['question'],
#                 'choices':           ex['choices'],
#                 'answer_key':        letter,
#                 'question_category': ex['question_category'],
#                 'hard_split':        self._normalize_hard_split(ex['hard_split']),
#             })
#         if max_samples:
#             samples = samples[:max_samples]
#         return samples


# # ============================================================
# # 3. Prompt 工具
# # ============================================================
# class PromptBuilder:
#     """
#     训练用：构造带答案的完整 prompt（供 SFTTrainer 做 causal LM 训练）
#     推理用：构造不含答案的 prompt（供 generate 输出）
#     """
#     @staticmethod
#     def build_train_prompt(sample: dict, max_scene_length: int) -> str:
#         scene   = sample['movie_scene'][:max_scene_length]
#         options = "\n".join(
#             f"{chr(65+i)}. {c}" for i, c in enumerate(sample['choices'])
#         )
#         answer  = sample['answer_key']
#         return (
#             f"You are watching a movie. Based on the scene description, "
#             f"answer the multiple choice question.\n\n"
#             f"Scene: {scene}\n\n"
#             f"Question: {sample['question']}\n\n"
#             f"Options:\n{options}\n\n"
#             f"Answer: {answer}"   # 训练时包含答案
#         )

#     @staticmethod
#     def build_inference_prompt(sample: dict, max_scene_length: int) -> str:
#         scene   = sample['movie_scene'][:max_scene_length]
#         options = "\n".join(
#             f"{chr(65+i)}. {c}" for i, c in enumerate(sample['choices'])
#         )
#         return (
#             f"You are watching a movie. Based on the scene description, "
#             f"answer the multiple choice question.\n\n"
#             f"Scene: {scene}\n\n"
#             f"Question: {sample['question']}\n\n"
#             f"Options:\n{options}\n\n"
#             f"Answer:"   # 推理时不含答案
#         )


# # ============================================================
# # 4. Evaluator（复用 Phase 1 逻辑）
# # ============================================================
# class Evaluator:
#     def __init__(self, config: Phase2Config):
#         self.config = config

#     def evaluate(self, model, tokenizer, test_data: list,
#                  method_name: str) -> dict:
#         model.eval()
#         records = []
#         for sample in tqdm(test_data, desc=f"  Evaluating {method_name}"):
#             pred = self._predict(model, tokenizer, sample)
#             records.append({
#                 'pred':              pred,
#                 'label':             sample['answer_key'],
#                 'question_category': sample['question_category'],
#                 'hard_split':        sample['hard_split'],
#             })
#         df = pd.DataFrame(records)
#         return self._compute_metrics(df)

#     def _predict(self, model, tokenizer, sample: dict) -> str:
#         prompt  = PromptBuilder.build_inference_prompt(
#             sample, self.config.max_scene_length
#         )
#         inputs  = tokenizer(
#             prompt,
#             return_tensors="pt",
#             truncation=True,
#             max_length=self.config.max_seq_length,
#         ).to(model.device)

#         with torch.no_grad():
#             outputs = model.generate(
#                 **inputs,
#                 max_new_tokens=self.config.max_new_tokens,
#                 do_sample=False,
#                 pad_token_id=tokenizer.eos_token_id,
#             )
#         decoded = tokenizer.decode(
#             outputs[0][inputs['input_ids'].shape[1]:],
#             skip_special_tokens=True,
#         ).strip().upper()

#         for char in decoded:
#             if char in 'ABCDE':
#                 return char
#         return 'A'

#     def _compute_metrics(self, df: pd.DataFrame) -> dict:
#         correct  = df['pred'] == df['label']
#         metrics  = {
#             'Overall':   correct.mean(),
#             'Overall_n': len(df),
#         }
#         for abbr, keyword in self.config.category_map.items():
#             mask = df['question_category'].str.contains(
#                 keyword, case=False, na=False
#             )
#             n = int(mask.sum())
#             metrics[abbr]        = correct[mask].mean() if n > 0 else None
#             metrics[f'{abbr}_n'] = n

#         hard_mask    = df['hard_split'] == True
#         n_hard       = int(hard_mask.sum())
#         metrics['Hard']   = correct[hard_mask].mean() if n_hard > 0 else None
#         metrics['Hard_n'] = n_hard
#         return metrics


# # ============================================================
# # 5. 基类 Trainer
# # ============================================================
# class BasePEFTTrainer(ABC):
#     def __init__(self, config: Phase2Config):
#         self.config    = config
#         self.model     = None
#         self.tokenizer = None
#         self.method_name = "base"

#     def _load_base_model_4bit(self):
#         """加载 4-bit 量化基础模型（QLoRA 用）"""
#         bnb_config = BitsAndBytesConfig(
#             load_in_4bit              = True,
#             bnb_4bit_use_double_quant = True,
#             bnb_4bit_quant_type       = "nf4",
#             bnb_4bit_compute_dtype    = torch.bfloat16,
#         )
#         tokenizer = AutoTokenizer.from_pretrained(self.config.base_model_id)
#         tokenizer.pad_token    = tokenizer.eos_token
#         tokenizer.padding_side = "right"

#         model = AutoModelForCausalLM.from_pretrained(
#             self.config.base_model_id,
#             quantization_config=bnb_config,
#             device_map="auto",
#         )
#         return model, tokenizer

#     def _load_base_model_bf16(self):
#         """加载 bf16 基础模型（Prefix Tuning / ReFT 用）"""
#         tokenizer = AutoTokenizer.from_pretrained(self.config.base_model_id)
#         tokenizer.pad_token    = tokenizer.eos_token
#         tokenizer.padding_side = "right"

#         model = AutoModelForCausalLM.from_pretrained(
#             self.config.base_model_id,
#             torch_dtype=torch.bfloat16,
#             device_map="auto",
#         )
#         return model, tokenizer

#     def _make_hf_dataset(self, data: list):
#         """将样本列表转为 HuggingFace Dataset（SFTTrainer 需要）"""
#         from datasets import Dataset
#         return Dataset.from_list([
#             {'text': PromptBuilder.build_train_prompt(
#                 s, self.config.max_scene_length
#             )}
#             for s in data
#         ])

#     def _output_dir(self) -> str:
#         path = os.path.join(self.config.output_base_dir, self.method_name)
#         os.makedirs(path, exist_ok=True)
#         return path

#     def free_memory(self):
#         if self.model is not None:
#             del self.model
#             self.model = None
#         if self.tokenizer is not None:
#             del self.tokenizer
#             self.tokenizer = None
#         gc.collect()
#         torch.cuda.empty_cache()
#         print(f"  [Memory freed] {self.method_name}")

#     @abstractmethod
#     def train(self, train_data: list): pass

#     @abstractmethod
#     def load_for_inference(self): pass


# # ============================================================
# # 6. QLoRA Trainer
# # ============================================================
# class QLoRATrainer(BasePEFTTrainer):
#     """
#     QLoRA: 4-bit 量化基础模型 + LoRA adapter
#     最成熟稳定，推理类问题（Causal / NPA）预期表现最好
#     """
#     def __init__(self, config: Phase2Config):
#         super().__init__(config)
#         self.method_name = f"qlora_r{config.lora_r}"

#     def train(self, train_data: list):
#         print(f"\n[QLoRA] Loading base model (4-bit)...")
#         model, tokenizer = self._load_base_model_4bit()

#         lora_config = LoraConfig(
#             r                = self.config.lora_r,
#             lora_alpha       = self.config.lora_alpha,
#             lora_dropout     = self.config.lora_dropout,
#             target_modules   = self.config.lora_target_modules,
#             bias             = "none",
#             task_type        = TaskType.CAUSAL_LM,
#         )

#         train_args = SFTConfig(
#             output_dir                  = self._output_dir(),
#             num_train_epochs            = self.config.num_train_epochs,
#             per_device_train_batch_size = self.config.per_device_train_batch_size,
#             gradient_accumulation_steps = self.config.gradient_accumulation_steps,
#             learning_rate               = self.config.learning_rate,
#             warmup_ratio                = self.config.warmup_ratio,
#             lr_scheduler_type           = self.config.lr_scheduler_type,
#             bf16                        = self.config.bf16,
#             logging_steps               = self.config.logging_steps,
#             save_steps                  = self.config.save_steps,
#             max_seq_length              = self.config.max_seq_length,
#             dataset_text_field          = "text",
#             report_to                   = "none",
#             gradient_checkpointing      = True,
#             optim                       = "paged_adamw_8bit",
#         )

#         hf_dataset = self._make_hf_dataset(train_data)

#         trainer = SFTTrainer(
#             model         = model,
#             args          = train_args,
#             train_dataset = hf_dataset,
#             peft_config   = lora_config,
#             tokenizer     = tokenizer,
#         )

#         print(f"[QLoRA] Training... (samples={len(train_data)})")
#         trainer.train()
#         trainer.save_model(self._output_dir())
#         tokenizer.save_pretrained(self._output_dir())
#         print(f"[QLoRA] Saved → {self._output_dir()}")

#         self.model     = trainer.model
#         self.tokenizer = tokenizer

#     def load_for_inference(self):
#         """从保存的 checkpoint 加载用于推理"""
#         print(f"[QLoRA] Loading from {self._output_dir()}")
#         model, tokenizer = self._load_base_model_4bit()
#         self.model     = PeftModel.from_pretrained(model, self._output_dir())
#         self.tokenizer = tokenizer


# # ============================================================
# # 7. Prefix Tuning Trainer
# # ============================================================
# class PrefixTuningTrainer(BasePEFTTrainer):
#     """
#     Prefix Tuning: 在每层添加可学习 virtual tokens
#     对长序列上下文任务（Temporal / CRD）预期有优势
#     注意：Prefix Tuning 不兼容 4-bit，需要 bf16 全精度基础模型
#     """
#     def __init__(self, config: Phase2Config):
#         super().__init__(config)
#         self.method_name = f"prefix_vt{config.prefix_num_virtual_tokens}"

#     def train(self, train_data: list):
#         print(f"\n[Prefix] Loading base model (bf16)...")
#         model, tokenizer = self._load_base_model_bf16()

#         prefix_config = PrefixTuningConfig(
#             task_type          = TaskType.CAUSAL_LM,
#             num_virtual_tokens = self.config.prefix_num_virtual_tokens,
#             prefix_projection  = True,   # MLP 投影，比直接优化更稳定
#         )

#         model = get_peft_model(model, prefix_config)
#         model.print_trainable_parameters()

#         train_args = SFTConfig(
#             output_dir                  = self._output_dir(),
#             num_train_epochs            = self.config.num_train_epochs,
#             per_device_train_batch_size = self.config.per_device_train_batch_size,
#             gradient_accumulation_steps = self.config.gradient_accumulation_steps,
#             learning_rate               = 1e-3,   # Prefix Tuning 需要更大 lr
#             warmup_ratio                = self.config.warmup_ratio,
#             lr_scheduler_type           = self.config.lr_scheduler_type,
#             bf16                        = self.config.bf16,
#             logging_steps               = self.config.logging_steps,
#             save_steps                  = self.config.save_steps,
#             max_seq_length              = self.config.max_seq_length,
#             dataset_text_field          = "text",
#             report_to                   = "none",
#             gradient_checkpointing      = False,  # Prefix Tuning 不兼容 grad ckpt
#         )

#         hf_dataset = self._make_hf_dataset(train_data)

#         trainer = SFTTrainer(
#             model         = model,
#             args          = train_args,
#             train_dataset = hf_dataset,
#             tokenizer     = tokenizer,
#         )

#         print(f"[Prefix] Training... (samples={len(train_data)})")
#         trainer.train()
#         trainer.save_model(self._output_dir())
#         tokenizer.save_pretrained(self._output_dir())
#         print(f"[Prefix] Saved → {self._output_dir()}")

#         self.model     = model
#         self.tokenizer = tokenizer

#     def load_for_inference(self):
#         print(f"[Prefix] Loading from {self._output_dir()}")
#         model, tokenizer = self._load_base_model_bf16()
#         self.model     = PeftModel.from_pretrained(model, self._output_dir())
#         self.tokenizer = tokenizer


# # ============================================================
# # 8. ReFT (LoReFT) Trainer
# # ============================================================
# class ReFTTrainer(BasePEFTTrainer):
#     """
#     ReFT / LoReFT: 在 representation 空间做低秩线性干预
#     参数极少 (<0.1%)，新方法，作为探索性对比
#     依赖: pip install pyreft
#     """
#     def __init__(self, config: Phase2Config):
#         super().__init__(config)
#         self.method_name = f"reft_r{config.reft_rank}"

#     def train(self, train_data: list):
#         try:
#             import pyreft
#         except ImportError:
#             print("[ReFT] pyreft not installed. Run: pip install pyreft")
#             return

#         print(f"\n[ReFT] Loading base model (bf16)...")
#         model, tokenizer = self._load_base_model_bf16()

#         # 在指定层应用 LoReFT 干预
#         reft_config = pyreft.ReftConfig(
#             representations=[
#                 {
#                     "layer"           : l,
#                     "component"       : "block_output",
#                     "low_rank_dimension": self.config.reft_rank,
#                     "intervention"    : pyreft.LoreftIntervention(
#                         embed_dim          = model.config.hidden_size,
#                         low_rank_dimension = self.config.reft_rank,
#                     )
#                 }
#                 for l in self.config.reft_layers
#             ]
#         )
#         reft_model = pyreft.get_reft_model(model, reft_config)
#         reft_model.print_trainable_parameters()

#         # 构造训练数据（ReFT 用自己的 DataCollator）
#         train_dataset = self._make_reft_dataset(
#             train_data, tokenizer, pyreft
#         )

#         training_args = TrainingArguments(
#             output_dir                  = self._output_dir(),
#             num_train_epochs            = self.config.num_train_epochs,
#             per_device_train_batch_size = self.config.per_device_train_batch_size,
#             gradient_accumulation_steps = self.config.gradient_accumulation_steps,
#             learning_rate               = self.config.learning_rate,
#             warmup_ratio                = self.config.warmup_ratio,
#             bf16                        = self.config.bf16,
#             logging_steps               = self.config.logging_steps,
#             save_steps                  = self.config.save_steps,
#             report_to                   = "none",
#         )

#         reft_trainer = pyreft.ReftTrainerForCausalLM(
#             model         = reft_model,
#             args          = training_args,
#             train_dataset = train_dataset,
#             tokenizer     = tokenizer,
#             data_collator = pyreft.ReftDataCollator(
#                 data_collator=DataCollatorWithPadding(tokenizer)
#             ),
#         )

#         print(f"[ReFT] Training... (samples={len(train_data)})")
#         reft_trainer.train()
#         reft_model.save(self._output_dir())
#         tokenizer.save_pretrained(self._output_dir())
#         print(f"[ReFT] Saved → {self._output_dir()}")

#         self.model     = reft_model
#         self.tokenizer = tokenizer

#     def _make_reft_dataset(self, data, tokenizer, pyreft):
#         """将样本转换为 ReFT 所需的 dataset 格式"""
#         from datasets import Dataset
#         prompts, outputs = [], []
#         for s in data:
#             prompt = PromptBuilder.build_inference_prompt(
#                 s, self.config.max_scene_length
#             )
#             prompts.append(prompt)
#             outputs.append(s['answer_key'])

#         return pyreft.make_last_position_supervised_data_module(
#             tokenizer     = tokenizer,
#             model         = self.model,
#             inputs        = prompts,
#             outputs       = outputs,
#             num_interventions = len(self.config.reft_layers),
#         )

#     def load_for_inference(self):
#         try:
#             import pyreft
#         except ImportError:
#             print("[ReFT] pyreft not installed.")
#             return

#         print(f"[ReFT] Loading from {self._output_dir()}")
#         model, tokenizer = self._load_base_model_bf16()
#         self.model     = pyreft.ReftModel.load(self._output_dir(), model)
#         self.tokenizer = tokenizer


# # ============================================================
# # 9. Phase2 主运行器
# # ============================================================
# class Phase2Runner:
#     def __init__(self, config: Phase2Config):
#         self.config      = config
#         self.dataset     = CinePileDataset(config)
#         self.evaluator   = Evaluator(config)
#         self.all_metrics = {}

#         # 注册要运行的 PEFT 方法
#         self.trainers = [
#             # QLoRATrainer(config),
#             # PrefixTuningTrainer(config),
#             ReFTTrainer(config),
#         ]

#     def run(self):
#         print("\n" + "="*60)
#         print("PHASE 2: PEFT Fine-tuning on CinePile")
#         print(f"Base model : {self.config.base_model_id}")
#         print(f"Train size : {len(self.dataset.train_data)}")
#         print(f"Test size  : {len(self.dataset.test_data)}")
#         print("="*60)

#         for trainer in self.trainers:
#             print(f"\n{'='*60}")
#             print(f"Method: {trainer.method_name.upper()}")
#             print(f"{'='*60}")

#             # 1. 训练
#             trainer.train(self.dataset.train_data)

#             # 2. 评估
#             if trainer.model is not None:
#                 metrics = self.evaluator.evaluate(
#                     trainer.model,
#                     trainer.tokenizer,
#                     self.dataset.test_data,
#                     trainer.method_name,
#                 )
#                 self.all_metrics[trainer.method_name] = metrics
#                 hard_str = (f"{metrics['Hard']:.2%}"
#                             if metrics.get('Hard') is not None else "N/A")
#                 print(f"  Overall: {metrics['Overall']:.2%} "
#                       f"| Hard: {hard_str}")

#             # 3. 释放显存
#             trainer.free_memory()

#         self._display_and_save()

#     def _display_and_save(self):
#         display_cols = ['Overall', 'TEMP', 'CRD', 'NPA', 'STA', 'TH', 'Hard']
#         rows = []
#         for method_name, metrics in self.all_metrics.items():
#             row = {'Method': method_name}
#             for col in display_cols:
#                 val = metrics.get(col)
#                 n   = metrics.get(f'{col}_n', metrics.get('Overall_n', ''))
#                 row[col] = f"{val:.1%}(n={n})" if val is not None else 'N/A'
#             rows.append(row)

#         df = pd.DataFrame(rows)
#         print("\n" + "="*60)
#         print("PHASE 2 FINAL RESULTS — CinePile PEFT Fine-tuning")
#         print("="*60)
#         print(df.to_string(index=False))
#         df.to_csv(self.config.output_csv, index=False)
#         print(f"\nSaved → {self.config.output_csv}")


# # ============================================================
# # 10. 入口
# # ============================================================
# if __name__ == '__main__':

#     config = Phase2Config(
#         base_model_id        = 'meta-llama/Llama-3.1-8B-Instruct',
#         max_train_samples    = 20,   # 调试：2000；完整：None
#         max_test_samples     = 500,    # 调试：500； 完整：None
#         num_train_epochs     = 2,
#         lora_r               = 16,
#         prefix_num_virtual_tokens = 20,
#         reft_rank            = 4,
#         reft_layers          = [8, 16, 24],
#         output_base_dir      = './phase2_outputs',
#         output_csv           = 'phase2_results.csv',
#     )

#     runner = Phase2Runner(config)
#     runner.run()


## 2.2 QLoRA and Prefix Tuning

In [ ]:
# ============================================================
# Phase 2: QLoRA + Prefix Tuning on CinePile (with Stratified Sampling & Checkpoints)
# Requirements:
#   pip install transformers datasets accelerate bitsandbytes peft trl
# ============================================================

import os
import gc
from dataclasses import dataclass, field
from typing import Optional, List, Dict
from datetime import datetime
from collections import defaultdict

import torch
import numpy as np
import pandas as pd
from datasets import load_dataset, Dataset
from tqdm import tqdm

torch.serialization.add_safe_globals([
    np.core.multiarray._reconstruct,
    np.ndarray,
    np.dtype,
    np.dtypes.UInt32DType
])

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
from peft import (
    LoraConfig,
    PrefixTuningConfig,
    get_peft_model,
    TaskType,
    PeftModel,
)
from trl import SFTTrainer, SFTConfig


# ============================================================
# 1. 配置
# ============================================================
@dataclass
class Phase2Config:
    # 数据集
    dataset_name: str          = "tomg-group-umd/cinepile"
    train_split: str           = "train"
    test_split: str            = "test"

    # 分层抽样（None = 用完整 split）
    max_train_samples: Optional[int] = 30000   # e.g. 从 ~30 万中抽 3 万
    max_test_samples: Optional[int]  = None     # 测试集可截断或用 full

    # hard 样本在分层抽样中的额外倍率（>1 表示更偏向抽 hard）
    hard_oversample_ratio: float     = 1.5   ### Increase hard sample oversampling, training needs stronger representation

    # 模型
    base_model_id: str         = "meta-llama/Llama-3.1-8B-Instruct"

    # 输入长度
    max_scene_length: int      = 1200   ### Change from 1000 to 1200 (allow more scene context)
    max_seq_length: int        = 1536   ### Change from 1024 to 1536 (better reasoning window)

    # 通用训练参数
    num_train_epochs: int      = 4      ### Change from 4 to 5 (LoRA rarely overfits, improves learning)
    per_device_train_batch_size: int = 4   ### Change from 4 to 1 (T4 VRAM stability)
    gradient_accumulation_steps: int = 4  ### Change from 4 to 16 (keep effective batch size ≈16)
    learning_rate: float       = 2e-4
    warmup_ratio: float        = 0.05
    lr_scheduler_type: str     = "cosine"

    bf16: bool                 = True   ### Change to True for A100
    fp16: bool                 = False

    logging_steps: int         = 50

    # checkpoint & 输出
    save_steps: int            = 200
    save_total_limit: int      = 3
    output_base_dir: str       = "./phase2_outputs"
    output_csv: str            = "phase2_results.csv"

    # QLoRA
    lora_r: int                = 64 ### Consider a higher value if the model is struggling to capture the complexity
    lora_alpha: int            = 128
    lora_dropout: float        = 0.00 ### Since we have big example sizes (30000), maybe try 0
    lora_target_modules: List[str] = field(default_factory=lambda: [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ]) ### Recommend all 7 layers, but might also try less

    # Prefix Tuning
    prefix_num_virtual_tokens: int = 20

    # 评估生成长度
    max_new_tokens: int        = 4

    # 类别映射
    category_map: Dict[str, str] = field(default_factory=lambda: {
        "TEMP": "Temporal",
        "CRD":  "Character and",
        "NPA":  "Narrative and",
        "STA":  "Setting and",
        "TH":   "Theme Exploration",
    })


# ============================================================
# 2. Stratified Sampler（按类别 × hard 分层抽样）
# ============================================================
class StratifiedSampler:
    def __init__(self, cfg: Phase2Config):
        self.cfg = cfg

    def sample(self, data: list) -> list:
        if self.cfg.max_train_samples is None:
            print(f"[Sampler] Using full train split: {len(data)} samples")
            return data

        target = self.cfg.max_train_samples

        # 按 (category_key, hard) 分组
        groups = defaultdict(list)
        for s in data:
            cat  = self._get_category_key(s["question_category"])
            hard = bool(s["hard_split"])
            groups[(cat, hard)].append(s)

        n_categories = len(set(k[0] for k in groups))
        # 理论 strata = 每个类别 × hard/easy
        base_quota = target / (n_categories * (1 + self.cfg.hard_oversample_ratio))
        easy_q     = base_quota
        hard_q     = base_quota * self.cfg.hard_oversample_ratio

        sampled = []
        report  = []

        import random
        random.seed(42) # same seed using in ReFT part

        for (cat, hard), group in sorted(groups.items()):
            quota  = int(hard_q if hard else easy_q)
            n_take = min(quota, len(group))
            taken  = random.sample(group, n_take)
            sampled.extend(taken)
            report.append({
                "category":  cat,
                "hard":      hard,
                "available": len(group),
                "quota":     quota,
                "sampled":   n_take,
            })

        # 若仍不足 target，从剩余中补齐
        if len(sampled) < target:
            sampled_ids = set(id(s) for s in sampled)
            remaining   = [s for s in data if id(s) not in sampled_ids]
            extra = random.sample(
                remaining, min(target - len(sampled), len(remaining))
            )
            sampled.extend(extra)

        self._print_report(report, len(sampled))
        return sampled

    def _get_category_key(self, cat_str: str) -> str:
        for abbr, kw in self.cfg.category_map.items():
            if kw.lower() in cat_str.lower():
                return abbr
        return "OTHER"

    @staticmethod
    def _print_report(report: list, total_sampled: int):
        df = pd.DataFrame(report)
        print("\n[Sampler] ===== Stratified Sampling Report =====")
        print(df.to_string(index=False))
        print(f"[Sampler] Total sampled: {total_sampled}")
        print("[Sampler] " + "=" * 60)


# ============================================================
# 3. ExperimentTracker（记录每次实验参数 + 结果）
# ============================================================
class ExperimentTracker:
    def __init__(self, log_path: str = "./phase2_experiments_log.csv"):
        self.log_path = log_path

    def log(self, method_name: str, cfg: Phase2Config, metrics: dict):
        record = {
            "timestamp":           datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "method":              method_name,
            "base_model":          cfg.base_model_id.split("/")[-1],
            "max_train_samples":   cfg.max_train_samples if cfg.max_train_samples else "full",
            "max_test_samples":    cfg.max_test_samples  if cfg.max_test_samples  else "full",
            "hard_oversample":     cfg.hard_oversample_ratio,
            "num_train_epochs":    cfg.num_train_epochs,
            "lora_r":              cfg.lora_r,
            "prefix_tokens":       cfg.prefix_num_virtual_tokens,
            "learning_rate":       cfg.learning_rate,
            "batch_size":          cfg.per_device_train_batch_size,
            "grad_accum":          cfg.gradient_accumulation_steps,
            # metrics
            "Overall": metrics.get("Overall"),
            "TEMP":    metrics.get("TEMP"),
            "CRD":     metrics.get("CRD"),
            "NPA":     metrics.get("NPA"),
            "STA":     metrics.get("STA"),
            "TH":      metrics.get("TH"),
            "Hard":    metrics.get("Hard"),
            "n_test":  metrics.get("Overall_n"),
            "n_TEMP":  metrics.get("TEMP_n"),
            "n_CRD":   metrics.get("CRD_n"),
            "n_NPA":   metrics.get("NPA_n"),
            "n_STA":   metrics.get("STA_n"),
            "n_TH":    metrics.get("TH_n"),
            "n_Hard":  metrics.get("Hard_n"),
        }
        df_new = pd.DataFrame([record])
        if os.path.exists(self.log_path):
            df_old = pd.read_csv(self.log_path)
            df_all = pd.concat([df_old, df_new], ignore_index=True)
        else:
            df_all = df_new
        df_all.to_csv(self.log_path, index=False)
        print(f"[Tracker] Logged {method_name} → {self.log_path}")
        return df_all


# ============================================================
# 4. 数据加载 & 预处理
# ============================================================
class CinePileDataset:
    LETTERS = ["A", "B", "C", "D", "E"]

    def __init__(self, cfg: Phase2Config):
        self.cfg = cfg
        self.train_data, self.test_data = self._load_all()
        print(f"[Dataset] Train: {len(self.train_data)} | Test: {len(self.test_data)}")

    @classmethod
    def _answer_text_to_letter(cls, answer_text: str, choices: list) -> str:
        for i, c in enumerate(choices):
            if answer_text.strip() == c.strip():
                return cls.LETTERS[i]
        for i, c in enumerate(choices):
            if answer_text.strip() in c or c.strip() in answer_text:
                return cls.LETTERS[i]
        return "A"

    @staticmethod
    def _normalize_hard_split(raw) -> bool:
        if isinstance(raw, bool):
            return raw
        return str(raw).strip().lower() == "true"

    def _load_split(self, split: str) -> list:
        raw = load_dataset(self.cfg.dataset_name, split=split)
        samples = []
        for ex in raw:
            letter = self._answer_text_to_letter(ex["answer_key"], ex["choices"])
            samples.append({
                "movie_scene":       ex["movie_scene"],
                "question":          ex["question"],
                "choices":           ex["choices"],
                "answer_key":        letter,
                "question_category": ex["question_category"],
                "hard_split":        self._normalize_hard_split(ex["hard_split"]),
            })
        return samples

    def _load_all(self):
        train_raw = self._load_split(self.cfg.train_split)
        test_raw  = self._load_split(self.cfg.test_split)

        # 训练集分层抽样
        sampler    = StratifiedSampler(self.cfg)
        train_data = sampler.sample(train_raw)

        # 测试集截断
        if self.cfg.max_test_samples:
            test_data = test_raw[:self.cfg.max_test_samples]
        else:
            test_data = test_raw

        return train_data, test_data


# ============================================================
# 5. Prompt Builder & Evaluator
# ============================================================
class PromptBuilder:
    @staticmethod
    def build_train_prompt(sample: dict, max_scene_length: int) -> str:
        scene   = sample["movie_scene"][:max_scene_length]
        options = "\n".join(
            f"{chr(65+i)}. {c}" for i, c in enumerate(sample["choices"])
        )
        return (
            "You are watching a movie. Based on the scene description, "
            "answer the multiple choice question.\n\n"
            f"Scene: {scene}\n\n"
            f"Question: {sample['question']}\n\n"
            f"Think briefly about the scene before choosing the answer.\n\n"
            f"Options:\n{options}\n\n"
            f"Answer (A/B/C/D/E): {sample['answer_key']}"
        )

    @staticmethod
    def build_inference_prompt(sample: dict, max_scene_length: int) -> str:
        scene   = sample["movie_scene"][:max_scene_length]
        options = "\n".join(
            f"{chr(65+i)}. {c}" for i, c in enumerate(sample["choices"])
        )
        return (
            "You are watching a movie. Based on the scene description, "
            "answer the multiple choice question.\n\n"
            f"Scene: {scene}\n\n"
            f"Question: {sample['question']}\n\n"
            f"Think briefly about the scene before choosing the answer.\n\n"
            f"Options:\n{options}\n\n"
            "Answer (A/B/C/D/E):"
        )


class Evaluator:
    def __init__(self, cfg: Phase2Config):
        self.cfg = cfg

    def evaluate(self, model, tokenizer, test_data: list, method_name: str) -> dict:
        model.eval()
        records = []
        for s in tqdm(test_data, desc=f"Eval {method_name}"):
            pred = self._predict(model, tokenizer, s)
            records.append({
                "pred":              pred,
                "label":             s["answer_key"],
                "question_category": s["question_category"],
                "hard_split":        s["hard_split"],
            })
        df = pd.DataFrame(records)
        return self._compute_metrics(df)

    def _predict(self, model, tokenizer, sample: dict) -> str:

        prompt = PromptBuilder.build_inference_prompt(
            sample,
            self.cfg.max_scene_length
        )

        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=self.cfg.max_seq_length,
        )

        inputs = {k: v.to(model.device) for k, v in inputs.items()} ## Add this line

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=self.cfg.max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )

        decoded = tokenizer.decode(
            outputs[0][inputs["input_ids"].shape[1]:],
            skip_special_tokens=True,
        ).strip().upper()

        import re

        match = re.search(r"\b([A-E])\b", decoded)
        if match:
            return match.group(1)

        return "A"

    def _compute_metrics(self, df: pd.DataFrame) -> dict:
        correct = df["pred"] == df["label"]
        metrics = {
            "Overall":   correct.mean(),
            "Overall_n": len(df),
        }
        for abbr, keyword in self.cfg.category_map.items():
            mask = df["question_category"].str.contains(keyword, case=False, na=False)
            n    = int(mask.sum())
            metrics[abbr]        = correct[mask].mean() if n > 0 else None
            metrics[f"{abbr}_n"] = n

        hard_mask = df["hard_split"] == True
        n_hard    = int(hard_mask.sum())
        metrics["Hard"]   = correct[hard_mask].mean() if n_hard > 0 else None
        metrics["Hard_n"] = n_hard
        return metrics


# ============================================================
# 6. 基类 Trainer
# ============================================================
class BasePEFTTrainer:
    def __init__(self, cfg: Phase2Config, method_name: str):
        self.cfg         = cfg
        self.method_name = method_name
        self.model       = None
        self.tokenizer   = None

    def _output_dir(self) -> str:
        path = os.path.join(self.cfg.output_base_dir, self.method_name)
        os.makedirs(path, exist_ok=True)
        return path

    def _make_hf_dataset(self, data: list) -> Dataset:
        return Dataset.from_list([
            {"text": PromptBuilder.build_train_prompt(s, self.cfg.max_scene_length)}
            for s in data
        ])

    def free_memory(self):
        if self.model is not None:
            del self.model
        if self.tokenizer is not None:
            del self.tokenizer
        self.model = None
        self.tokenizer = None
        gc.collect()
        torch.cuda.empty_cache()
        print(f"[Memory] Freed for {self.method_name}")


# ============================================================
# 7. QLoRA Trainer（含 checkpoint）
# ============================================================
class QLoRATrainer(BasePEFTTrainer):
    def __init__(self, cfg: Phase2Config):
        super().__init__(cfg, method_name=f"qlora_r{cfg.lora_r}")

    def _load_base_model_4bit(self):

        bnb_config = BitsAndBytesConfig(
            load_in_4bit              = True,
            bnb_4bit_use_double_quant = True,
            bnb_4bit_quant_type       = "nf4",

            bnb_4bit_compute_dtype=torch.bfloat16

            ### Recover bfloat16 for A100

        )

        tokenizer = AutoTokenizer.from_pretrained(self.cfg.base_model_id)

        tokenizer.pad_token    = tokenizer.eos_token
        tokenizer.padding_side = "right"

        model = AutoModelForCausalLM.from_pretrained(
            self.cfg.base_model_id,
            quantization_config=bnb_config,
            device_map="auto",
        )

        return model, tokenizer

    def train(self, train_data: list, resume_checkpoint: Optional[str] = None):
        print(f"\n[QLoRA] Loading base model (4-bit)...")
        model, tokenizer = self._load_base_model_4bit()

        lora_config = LoraConfig(
            r              = self.cfg.lora_r,
            lora_alpha     = self.cfg.lora_alpha,
            lora_dropout   = self.cfg.lora_dropout,
            target_modules = self.cfg.lora_target_modules,
            bias           = "none",
            task_type      = TaskType.CAUSAL_LM,
        )

        args = SFTConfig(
            output_dir                  = self._output_dir(),
            num_train_epochs            = self.cfg.num_train_epochs,
            per_device_train_batch_size = self.cfg.per_device_train_batch_size,
            gradient_accumulation_steps = self.cfg.gradient_accumulation_steps,
            learning_rate               = self.cfg.learning_rate,
            warmup_ratio                = self.cfg.warmup_ratio,
            lr_scheduler_type           = self.cfg.lr_scheduler_type,

            bf16=self.cfg.bf16,
            fp16=self.cfg.fp16,
            ### Added fp16=True support for T4 GPU

            logging_steps               = self.cfg.logging_steps,

            save_strategy               = "steps",
            save_steps                  = self.cfg.save_steps,
            save_total_limit            = self.cfg.save_total_limit,
            max_seq_length              = self.cfg.max_seq_length, # Move to SFTTrainer
            # dataset_text_field          = "text",
            report_to                   = "none",
            gradient_checkpointing      = True,
            optim                       = "paged_adamw_8bit",
        )

        hf_dataset = self._make_hf_dataset(train_data)
        trainer = SFTTrainer(
            model         = model,
            args          = args,
            train_dataset = hf_dataset,
            peft_config   = lora_config,
            tokenizer     = tokenizer,


            dataset_text_field = 'text', # Improve GPU utilization
            packing       = False, # Improve GPU utilization

        )

        trainer.model.to(torch.bfloat16)

        latest_ckpt = get_latest_checkpoint(self._output_dir())

        if latest_ckpt:
            print(f"[QLoRA] Resuming from {latest_ckpt}")
            trainer.train(resume_from_checkpoint=latest_ckpt)
        else:
            print("[QLoRA] Training from scratch")
            trainer.train()

        trainer.save_model(self._output_dir())
        tokenizer.save_pretrained(self._output_dir())
        print(f"[QLoRA] Saved → {self._output_dir()}")

        self.model     = trainer.model
        self.tokenizer = tokenizer


# # ============================================================
# # 8. Prefix Tuning Trainer（含 checkpoint）
# # ============================================================
# class PrefixTuningTrainer(BasePEFTTrainer):
#     def __init__(self, cfg: Phase2Config):
#         super().__init__(cfg, method_name=f"prefix_vt{cfg.prefix_num_virtual_tokens}")

#     def _load_base_model_bf16(self):
#         tokenizer = AutoTokenizer.from_pretrained(self.cfg.base_model_id)
#         tokenizer.pad_token    = tokenizer.eos_token
#         tokenizer.padding_side = "right"
#         model = AutoModelForCausalLM.from_pretrained(
#             self.cfg.base_model_id,
#             torch_dtype=torch.bfloat16,
#             device_map="auto",
#         )
#         return model, tokenizer

#     def train(self, train_data: list):
#         print(f"\n[Prefix] Loading base model (bf16)...")
#         model, tokenizer = self._load_base_model_bf16()

#         prefix_config = PrefixTuningConfig(
#             task_type          = TaskType.CAUSAL_LM,
#             num_virtual_tokens = self.cfg.prefix_num_virtual_tokens,
#             prefix_projection  = True,
#         )
#         model = get_peft_model(model, prefix_config)
#         model.print_trainable_parameters()

#         args = SFTConfig(
#             output_dir                  = self._output_dir(),
#             num_train_epochs            = self.cfg.num_train_epochs,
#             per_device_train_batch_size = self.cfg.per_device_train_batch_size,
#             gradient_accumulation_steps = self.cfg.gradient_accumulation_steps,
#             learning_rate               = 1e-3,   # Prefix 一般 lr 更大
#             warmup_ratio                = self.cfg.warmup_ratio,
#             lr_scheduler_type           = self.cfg.lr_scheduler_type,
#             bf16                        = self.cfg.bf16,
#             logging_steps               = self.cfg.logging_steps,
#             save_strategy               = "steps",
#             save_steps                  = self.cfg.save_steps,
#             save_total_limit            = self.cfg.save_total_limit,
#             max_seq_length              = self.cfg.max_seq_length,
#             dataset_text_field          = "text",
#             report_to                   = "none",
#             gradient_checkpointing      = False,
#         )

#         hf_dataset = self._make_hf_dataset(train_data)
#         trainer = SFTTrainer(
#             model         = model,
#             args          = args,
#             train_dataset = hf_dataset,
#             tokenizer     = tokenizer,
#         )

#         print(f"[Prefix] Training on {len(train_data)} samples...")
#         trainer.train()
#         trainer.save_model(self._output_dir())
#         tokenizer.save_pretrained(self._output_dir())
#         print(f"[Prefix] Saved → {self._output_dir()}")

#         self.model     = trainer.model
#         self.tokenizer = tokenizer


# ============================================================
# 9. Phase2 Runner
# ============================================================
class Phase2Runner:
    def __init__(self, cfg: Phase2Config, tracker: Optional[ExperimentTracker] = None):
        self.cfg      = cfg
        self.dataset  = CinePileDataset(cfg)
        self.evaluator = Evaluator(cfg)
        self.tracker  = tracker
        self.results  = {}

        self.trainers = [
            QLoRATrainer(cfg),
            # PrefixTuningTrainer(cfg),
        ]

    def run(self):
        print("\n" + "=" * 70)
        print("PHASE 2: QLoRA + Prefix Tuning with Stratified Sampling")
        print(f"Base model : {self.cfg.base_model_id}")
        print(f"Train size : {len(self.dataset.train_data)} (after sampling)")
        print(f"Test size  : {len(self.dataset.test_data)}")
        print("=" * 70)

        for trainer in self.trainers:
            print("\n" + "=" * 70)
            print(f"Method: {trainer.method_name.upper()}")
            print("=" * 70)

            trainer.train(self.dataset.train_data)

            merged_model, merged_tokenizer = load_merged_model(
                self.cfg.base_model_id,
                trainer._output_dir()
            )

            metrics = self.evaluator.evaluate(
                merged_model,
                merged_tokenizer,
                self.dataset.test_data,
                trainer.method_name,
            )

            self.results[trainer.method_name] = metrics

            hard_str = (f"{metrics['Hard']:.2%}"
                        if metrics.get("Hard") is not None else "N/A")
            print(f"[{trainer.method_name}] Overall: {metrics['Overall']:.2%} | Hard: {hard_str}")

            if self.tracker is not None:
                self.tracker.log(trainer.method_name, self.cfg, metrics)

            trainer.free_memory()

            del merged_model
            del merged_tokenizer
            gc.collect()
            torch.cuda.empty_cache()

        self._display_and_save()

    def _display_and_save(self):
        display_cols = ["Overall", "TEMP", "CRD", "NPA", "STA", "TH", "Hard"]
        rows = []
        for method, metrics in self.results.items():
            row = {"Method": method}
            for col in display_cols:
                val = metrics.get(col)
                n   = metrics.get(f"{col}_n", metrics.get("Overall_n", ""))
                row[col] = f"{val:.1%}(n={n})" if val is not None else "N/A"
            rows.append(row)

        df = pd.DataFrame(rows)
        print("\n" + "=" * 70)
        print("PHASE 2 FINAL RESULTS — QLoRA + Prefix (Stratified 30k)")
        print("=" * 70)
        print(df.to_string(index=False))
        df.to_csv(self.cfg.output_csv, index=False)
        print(f"\n[Phase2] Saved table → {self.cfg.output_csv}")



def load_merged_model(base_model_id: str, adapter_path:str):

    print("[Merge] Loading base model...")

    base_model = AutoModelForCausalLM.from_pretrained(
        base_model_id,
        torch_dtype=torch.bfloat16,
        device_map="auto",
    )

    tokenizer = AutoTokenizer.from_pretrained(base_model_id)

    print("[Merge] Loading LoRA adapter...")
    model = PeftModel.from_pretrained(base_model, adapter_path)

    print("[Merge] Merging LoRA weights...")
    model = model.merge_and_unload()

    model.eval()

    return model, tokenizer


def get_latest_checkpoint(output_dir: str):

    if not os.path.exists(output_dir):
        return None

    checkpoints = [
        os.path.join(output_dir, d)
        for d in os.listdir(output_dir)
        if d.startswith("checkpoint")
    ]

    if not checkpoints:
        return None

    checkpoints = sorted(
        checkpoints,
        key=lambda x: int(x.split("checkpoint-")[-1])
    )

    return checkpoints[-1]


In [ ]:
if __name__ == "__main__":

    tracker = ExperimentTracker(
        log_path=os.path.join(PROJECT_DIR, "phase2_experiments_log.csv")
    )

    cfg = Phase2Config(
        base_model_id               = "meta-llama/Llama-3.1-8B-Instruct",
        max_train_samples           = 30000,    # 从约30万中抽3万
        max_test_samples            = None,
        hard_oversample_ratio       = 1.5,

        num_train_epochs            = 4,
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 4,

        learning_rate               = 2e-4,
        output_base_dir             = PROJECT_DIR,
        output_csv                  = os.path.join(PROJECT_DIR, "phase2_results.csv"),
    )

    runner = Phase2Runner(cfg, tracker=tracker)
    runner.run()


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

[Sampler] ===== Stratified Sampling Report =====
category  hard  available  quota  sampled
     CRD False     127468   2400     2400
     NPA False      32238   2400     2400
     STA False      85604   2400     2400
    TEMP False      41517   2400     2400
      TH False      12061   2400     2400
[Sampler] Total sampled: 30000
[Sampler] ============================================================
[Dataset] Train: 30000 | Test: 4941

PHASE 2: QLoRA + Prefix Tuning with Stratified Sampling
Base model : meta-llama/Llama-3.1-8B-Instruct
Train size : 30000 (after sampling)
Test size  : 4941

Method: QLORA_R64

[QLoRA] Loading base model (4-bit)...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_deprecation.py:100: FutureWarning: Deprecated argument(s) used in '__init__': dataset_text_field. Will not be supported from version '1.0.0'.

Deprecated positional argument(s) used in SFTTrainer, please use the SFTConfig to set these arguments instead.
  warnings.warn(message, FutureWarning)
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:318: UserWarning: You passed a `dataset_text_field` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(


Map:   0%|          | 0/30000 [00:00<?, ? examples/s]

[QLoRA] Resuming from /content/drive/MyDrive/Datasci_266_NLP/Final_Project/CinePile_Phase2/qlora_r64/checkpoint-2200


...

[QLoRA] Saved → /content/drive/MyDrive/Datasci_266_NLP/Final_Project/CinePile_Phase2/qlora_r64
[Merge] Loading base model...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

[Merge] Loading LoRA adapter...
[Merge] Merging LoRA weights...


Eval qlora_r64:   0%|          | 0/4941 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:601: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:606: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
Starting from v4.46, the `logits` model output will have the same type as the model (except at train time, where it will always be FP32)
Eval qlora_r64: 100%|██████████| 4941/4941 [14:04<00:00,  5.85it/s]


[qlora_r64] Overall: 68.73% | Hard: 49.83%
[Tracker] Logged qlora_r64 → /content/drive/MyDrive/Datasci_266_NLP/Final_Project/CinePile_Phase2/phase2_experiments_log.csv
[Memory] Freed for qlora_r64

PHASE 2 FINAL RESULTS — QLoRA + Prefix (Stratified 30k)
   Method       Overall         TEMP           CRD          NPA           STA           TH         Hard
qlora_r64 68.7%(n=4941) 56.0%(n=688) 70.3%(n=2068) 70.6%(n=463) 72.1%(n=1532) 65.8%(n=190) 49.8%(n=887)

[Phase2] Saved table → /content/drive/MyDrive/Datasci_266_NLP/Final_Project/CinePile_Phase2/phase2_results.csv


In [ ]:
# ============================================================
# 诊断代码：直接粘贴到 Colab 运行
# ============================================================
from datasets import load_dataset

dataset = load_dataset('tomg-group-umd/cinepile', split='test')

# Step 1: 查看原始字段
sample = dataset[0]
print("=== 所有字段名 ===")
print(list(sample.keys()))

print("\n=== answer_key 值和类型 ===")
print(repr(sample['answer_key']))
print(type(sample['answer_key']))

print("\n=== choices 值和类型 ===")
print(sample['choices'])
print(type(sample['choices'][0]))

print("\n=== question_category 示例 ===")
print(repr(sample['question_category']))

print("\n=== hard_split 值和类型 ===")
print(repr(sample['hard_split']))
print(type(sample['hard_split']))

# Step 2: 查看前5个样本的 answer_key
print("\n=== 前5个样本的 answer_key ===")
for i in range(5):
    print(f"  [{i}] answer_key = {repr(dataset[i]['answer_key'])}")

# Step 3: 查看所有 answer_key 的唯一值
print("\n=== answer_key 的所有唯一值（前100个样本）===")
unique_keys = set(dataset[i]['answer_key'] for i in range(100))
print(unique_keys)

# Step 4: 查看 question_category 的唯一值
print("\n=== question_category 的唯一值（前100个样本）===")
unique_cats = set(dataset[i]['question_category'] for i in range(100))
for c in sorted(unique_cats):
    print(f"  {repr(c)}")

# Step 5: 手动测试 DeBERTa predict 一次
print("\n=== DeBERTa 手动预测测试 ===")
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained('microsoft/deberta-v3-base')
model = AutoModelForSequenceClassification.from_pretrained(
    'microsoft/deberta-v3-base',
    num_labels=1
)
model.eval()

test_sample = dataset[0]
scores = []
scene_q = f"{test_sample['question']} {test_sample['movie_scene'][:512]}"

for i, choice in enumerate(test_sample['choices']):
    inputs = tokenizer(
        scene_q, choice,
        return_tensors='pt',
        truncation=True,
        max_length=512,
        padding=True
    )
    with torch.no_grad():
        logit = model(**inputs).logits.squeeze().item()
    scores.append(logit)
    print(f"  Choice {chr(65+i)}: logit = {logit:.4f}")

pred = chr(65 + scores.index(max(scores)))
label = test_sample['answer_key']
print(f"\n  Predicted: {pred}")
print(f"  Label:     {repr(label)}")
print(f"  Match:     {pred == label}")


=== 所有字段名 ===
['movie_name', 'year', 'genre', 'yt_clip_title', 'yt_clip_link', 'movie_scene', 'subtitles', 'question', 'choices', 'answer_key', 'answer_key_position', 'question_category', 'hard_split', 'visual_reliance', 'videoID']

=== answer_key 值和类型 ===
'From anxiety to excitement'
<class 'str'>

=== choices 值和类型 ===
['From despair to hope', 'From fear to acceptance', 'From confusion to understanding', 'From tension to panic', 'From anxiety to excitement']
<class 'str'>

=== question_category 示例 ===
'Theme Exploration'

=== hard_split 值和类型 ===
'True'
<class 'str'>

=== 前5个样本的 answer_key ===
  [0] answer_key = 'From anxiety to excitement'
  [1] answer_key = 'Suggests next steps'
  [2] answer_key = 'The rough foliage'
  [3] answer_key = 'She closes her eyes and focuses on staying calm without moving at all.'
  [4] answer_key = 'Rough and dense'

=== answer_key 的所有唯一值（前100个样本）===
{'Throw jerky to the dog, or it will injure people.', 'She visibly trembles but stays still.', 'Directly un

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:558: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  Choice A: logit = -0.0255
  Choice B: logit = -0.0253
  Choice C: logit = -0.0259
  Choice D: logit = -0.0254
  Choice E: logit = -0.0254

  Predicted: B
  Label:     'From anxiety to excitement'
  Match:     False


In [ ]:
from datasets import load_dataset

dataset = load_dataset('tomg-group-umd/cinepile', split='test')

# 查看前20个 answer_key 的原始值
print("=== 前20个 answer_key ===")
for i in range(20):
    ak = dataset[i]['answer_key']
    print(f"  [{i}] type={type(ak).__name__}, value={repr(ak)}")

# 查看所有唯一值
print("\n=== 所有唯一 answer_key 值（完整测试集）===")
from collections import Counter
counter = Counter(str(dataset[i]['answer_key']) for i in range(len(dataset)))
for val, cnt in sorted(counter.items()):
    print(f"  {repr(val):10s} → {cnt} 条")


=== 前20个 answer_key ===
  [0] type=str, value='From anxiety to excitement'
  [1] type=str, value='Suggests next steps'
  [2] type=str, value='The rough foliage'
  [3] type=str, value='She closes her eyes and focuses on staying calm without moving at all.'
  [4] type=str, value='Rough and dense'
  [5] type=str, value='They hear an aircraft approaching.'
  [6] type=str, value='To reach a specific location'
  [7] type=str, value='They become more confident'
  [8] type=str, value='The sound of a helicopter approaching'
  [9] type=str, value='Reed'
  [10] type=str, value='The sound of a helicopter'
  [11] type=str, value='In a rocky plateau'
  [12] type=str, value='It is a valley with spots of light'
  [13] type=str, value='They use a thermal suit'
  [14] type=str, value='They are trying to remain undetected by the helicopter using thermal suits.'
  [15] type=str, value='It illuminates the surrounding foliage.'
  [16] type=str, value='Directly underneath'
  [17] type=str, value='Colleagues'

In [ ]:
# 运行这段诊断，查看实际的 question_category 值分布
from datasets import load_dataset
from collections import Counter

# dataset = load_dataset('tomg-group-umd/cinepile', split='test')

counter = Counter(dataset[i]['question_category'] for i in range(200))
print("=== question_category 实际值分布 ===")
for val, cnt in sorted(counter.items(), key=lambda x: -x[1]):
    print(f"  {cnt:4d}  {repr(val)}")


=== question_category 实际值分布 ===
    88  'Character and\nRelationship Dynamics'
    63  'Setting and\nTechnical Analysis'
    24  'Temporal'
    13  'Narrative and\nPlot Analysis'
    12  'Theme Exploration'


In [ ]:
# 快速验证（无需重新加载数据集）
test_categories = [
    'Character and\nRelationship Dynamics',
    'Setting and\nTechnical Analysis',
    'Temporal',
    'Narrative and\nPlot Analysis',
    'Theme Exploration',
]

category_map = {
    'TEMP': 'Temporal',
    'CRD':  'Character and',
    'NPA':  'Narrative and',
    'STA':  'Setting and',
    'TH':   'Theme Exploration',
}

import pandas as pd
df_test = pd.DataFrame({'question_category': test_categories})

print("=== 映射验证 ===")
for abbr, keyword in category_map.items():
    mask = df_test['question_category'].str.contains(keyword, case=False, na=False)
    matched = df_test.loc[mask, 'question_category'].tolist()
    print(f"  {abbr} ({keyword!r:20s}) → {matched}")


=== 映射验证 ===
  TEMP ('Temporal'          ) → ['Temporal']
  CRD ('Character and'     ) → ['Character and\nRelationship Dynamics']
  NPA ('Narrative and'     ) → ['Narrative and\nPlot Analysis']
  STA ('Setting and'       ) → ['Setting and\nTechnical Analysis']
  TH ('Theme Exploration' ) → ['Theme Exploration']
